# Meridian — Financial Diffusion Augmentation (FDA) Generation (Colab)

Runs the FDA offline pipeline (§8.2–8.3 of `meridian-spec.md`):
1. Trains a regime-conditioned diffusion model on real 100-asset windows.
2. Generates synthetic windows and gates them with the 9-point stylized-fact check.
3. Writes accepted windows as `FDA_SYNTH_*.f32bin` shards into the training data dir.
4. Uploads the new shards to the HuggingFace dataset so portfolio training can auto-detect them.

**Run this before `colab_portfolio_return.ipynb`.**
`colab_portfolio_return.ipynb` and `colab_portfolio_sharpe.ipynb` automatically mix in any `FDA_SYNTH_*.f32bin` shards they find in the data dir — no code changes needed on their side.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
HF_USERNAME   = 'Verdenroz'
HF_TOKEN      = ''            # HF token with WRITE access (required for shard upload)
                              # create at: huggingface.co/settings/tokens
HF_MODEL_REPO = f'{HF_USERNAME}/meridian-checkpoints'
HF_DATA_REPO  = f'{HF_USERNAME}/meridian-data'

# Generation target — spec §11.4c calls for 50,000 valid windows.
# Start smaller (5,000–10,000) to verify the pipeline end-to-end, then scale up.
TARGET_VALID_WINDOWS = 50_000

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/meridian'
os.makedirs(f'{DRIVE_DIR}/data/mmaps',      exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/fda',        exist_ok=True)
print('Drive ready:', DRIVE_DIR)

In [ ]:
# ── 2. HuggingFace login + install deps ───────────────────────────────────────────────
import subprocess, sys
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'pyyaml>=6.0',
    'hmmlearn>=0.3', 'faiss-cpu>=1.8', 'tokenizers>=0.20',
    'huggingface_hub>=0.26',
], check=True)

from huggingface_hub import login, snapshot_download, hf_hub_download, HfApi
if HF_TOKEN:
    login(token=HF_TOKEN)
print('HuggingFace ready.')

In [ ]:
# ── 3. Download training code + symbols file ──────────────────────────────────────────
import os, shutil

if not os.path.exists('/content/meridian'):
    local = snapshot_download(repo_id=HF_MODEL_REPO, allow_patterns='code/**')
    shutil.copytree(os.path.join(local, 'code'), '/content/meridian')
    print('Code downloaded.')
else:
    print('Code already present.')

symbols_path = '/content/meridian/symbols.txt'
if not os.path.exists(symbols_path):
    path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='symbols.txt')
    shutil.copy(path, symbols_path)
    print(f'symbols.txt downloaded → {symbols_path}')
else:
    print('symbols.txt already present.')

In [ ]:
# ── 4. Download real data (cached to Drive across sessions) ─────────────────────────────────
import os, shutil

data_dir  = f'{DRIVE_DIR}/data/mmaps'
train_dir = f'{data_dir}/train'
has_data  = os.path.exists(train_dir) and any(f.endswith('.f32bin') for f in os.listdir(train_dir))

if not has_data:
    snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=data_dir)
    for prefix in [os.path.join(data_dir, 'data', 'mmaps'), os.path.join(data_dir, 'mmaps')]:
        if os.path.exists(os.path.join(prefix, 'train')):
            for item in os.listdir(prefix):
                src = os.path.join(prefix, item)
                dst = os.path.join(data_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(os.path.join(data_dir, prefix.replace(data_dir + '/', '').split('/')[0]), ignore_errors=True)
            break
    print(f'Data downloaded → {data_dir}')
else:
    print('Data already in Drive.')

In [ ]:
# ── 5. Copy data to local SSD for fast I/O ───────────────────────────────────────────────
import shutil, os

local_data = '/content/data/mmaps'
if not os.path.exists(f'{local_data}/train'):
    print('Copying data from Drive to local SSD (one-time per session)...')
    shutil.copytree(f'{DRIVE_DIR}/data/mmaps', local_data)
    print('Done.')
else:
    print('Local data already present.')

real_shards = sorted(f for f in os.listdir(f'{local_data}/train') if f.endswith('.f32bin') and not f.startswith('FDA_SYNTH_'))
synth_shards_existing = sorted(f for f in os.listdir(f'{local_data}/train') if f.startswith('FDA_SYNTH_'))
print(f'Real train shards: {len(real_shards)}  |  Existing FDA synth shards: {len(synth_shards_existing)}')

In [ ]:
# ── 6. GPU detection ─────────────────────────────────────────────────────────────────────
import torch, os
assert torch.cuda.is_available(), 'No GPU — enable it in Runtime → Change runtime type.'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB VRAM)')

# FDA diffusion model is lightweight (~1–2 GB VRAM at batch=128). Any GPU works.
if vram_gb >= 35:
    FDA_BATCH_SIZE        = 256
    FDA_GEN_BATCH_SIZE    = 256
    FDA_MAX_REAL_WINDOWS  = 50_000
elif vram_gb >= 22:
    FDA_BATCH_SIZE        = 128
    FDA_GEN_BATCH_SIZE    = 128
    FDA_MAX_REAL_WINDOWS  = 30_000
else:
    FDA_BATCH_SIZE        = 64
    FDA_GEN_BATCH_SIZE    = 64
    FDA_MAX_REAL_WINDOWS  = 20_000

print(f'fda_batch={FDA_BATCH_SIZE}  gen_batch={FDA_GEN_BATCH_SIZE}  max_real_windows={FDA_MAX_REAL_WINDOWS}')

In [ ]:
# ── 7. Run FDA pipeline ──────────────────────────────────────────────────────────────
# The pipeline writes FDA_SYNTH_NNNNN.f32bin shards directly into the same train
# directory as the real shards — portfolio_train.py scans for the FDA_SYNTH_
# prefix and auto-mixes them in (see portfolio_train.py lines ~2256–2258).
import os, sys, subprocess

OUTPUT_DIR = '/content/data/mmaps'  # same dir as real shards; writes to OUTPUT_DIR/train/

cmd = [
    sys.executable, '-u',
    '-m', 'train.fda_pipeline',
    '--input-dir',             f'{OUTPUT_DIR}/train',
    '--output-dir',            OUTPUT_DIR,
    '--max-real-windows',      str(FDA_MAX_REAL_WINDOWS),
    '--window-len',            '64',
    '--stride',                '4',
    '--epochs',                '10',
    '--batch-size',            str(FDA_BATCH_SIZE),
    '--target-valid-windows',  str(TARGET_VALID_WINDOWS),
    '--device',                'cuda',
]

env = os.environ.copy()
env['PYTHONPATH']       = '/content/meridian'
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(cmd))
print('-' * 80)

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'FDA pipeline failed with return code {process.returncode}')
print('FDA generation complete.')

In [ ]:
# ── 8. Mirror generated shards + manifest to Drive (for session resume) ────────────
import os, shutil

src_train = '/content/data/mmaps/train'
dst_train = f'{DRIVE_DIR}/data/mmaps/train'

synth_files = sorted(f for f in os.listdir(src_train) if f.startswith('FDA_SYNTH_'))
print(f'Mirroring {len(synth_files)} FDA shards to Drive...')
for i, f in enumerate(synth_files):
    src = os.path.join(src_train, f)
    dst = os.path.join(dst_train, f)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
    if (i + 1) % 1000 == 0:
        print(f'  {i + 1}/{len(synth_files)}')

manifest_src = '/content/data/mmaps/fda_manifest.json'
if os.path.exists(manifest_src):
    shutil.copy2(manifest_src, f'{DRIVE_DIR}/data/fda/fda_manifest.json')
    print('Manifest mirrored to Drive.')
print('Drive sync done.')

In [ ]:
# ── 9. Upload FDA shards + manifest to HuggingFace dataset ────────────────────────────
# Uploads the FDA_SYNTH_*.f32bin shards into the meridian-data dataset so future
# Colab sessions (and colab_portfolio_return.ipynb) pick them up automatically
# via snapshot_download.
import os
from huggingface_hub import HfApi

api = HfApi()
src_train = '/content/data/mmaps/train'
synth_files = sorted(f for f in os.listdir(src_train) if f.startswith('FDA_SYNTH_'))

if not synth_files:
    print('No FDA shards to upload.')
else:
    # upload_folder with allow_patterns is much faster than per-file upload_file.
    print(f'Uploading {len(synth_files)} FDA shards to {HF_DATA_REPO} ...')
    api.upload_folder(
        folder_path=src_train,
        path_in_repo='train',
        repo_id=HF_DATA_REPO,
        repo_type='dataset',
        allow_patterns=['FDA_SYNTH_*.f32bin'],
        commit_message=f'Add {len(synth_files)} FDA synthetic shards',
    )
    print(f'Uploaded {len(synth_files)} shards → {HF_DATA_REPO}/train/FDA_SYNTH_*.f32bin')

manifest_src = '/content/data/mmaps/fda_manifest.json'
if os.path.exists(manifest_src):
    api.upload_file(
        path_or_fileobj=manifest_src,
        path_in_repo='fda_manifest.json',
        repo_id=HF_DATA_REPO,
        repo_type='dataset',
        commit_message='Update FDA manifest',
    )
    print(f'Manifest uploaded → {HF_DATA_REPO}/fda_manifest.json')